# Task 1.3: What the Paper Claims to Improve
**Paper**: *Ensemble of Exemplar-SVMs for Object Detection and Beyond* — Malisiewicz, Gupta & Efros (ICCV 2011)

---

## Main Baseline Method

The primary baseline the paper compares against is the **Dalal-Triggs HOG detector** [4] — a single linear SVM trained on all positive examples of a category using HOG features. On PASCAL VOC 2007, the paper also compares against the **deformable part-based model (DPM) of Felzenszwalb et al.** [9]. Both are category-level detectors: they learn one (or a few) template(s) per object category from all available positive training examples.

## Limitation of the Baseline

The Dalal-Triggs detector learns a **single average template** per category by training one linear SVM on all positives. This averaging smooths away the within-category variation — a 'person' category includes people standing, sitting, running, in different clothing, at different scales. The resulting template is a blurry average that matches all instances poorly rather than matching any instance well. As the paper states, this averaging causes a *'washing-out of features'* (Section 2) across the many different appearances within a category. Consequently, the single-template detector has low precision: it cannot distinguish subtle visual differences between objects and produces many false positives on cluttered backgrounds. The DPM partially addresses this with mixture components and deformable parts, but still averages within each mixture, and the number of mixtures is fixed.

## How the Proposed Method Overcomes This

The Exemplar-SVM approach overcomes this limitation by **training a separate detector per training instance rather than averaging across the category**. Each Exemplar-SVM learns a weight vector $w$ that is specific to one positive example, capturing the fine-grained appearance details of that particular instance (its specific viewpoint, pose, and appearance). The ensemble of these instance-specific detectors collectively covers the full within-category variation without ever averaging. Additionally, because each detection is associated with a specific training exemplar, the method enables **direct meta-data transfer** (segmentation masks, 3D geometry, viewpoint labels) from the matched exemplar to the detection — something that category-level detectors like Dalal-Triggs simply cannot do because detections are not linked to specific training instances.

## When the Paper's Method Would NOT Outperform the Baseline

The Exemplar-SVM method would likely **not** outperform the Dalal-Triggs baseline when the target category has very **low within-class visual variation** and the dataset has very **few positive training examples**. Consider detecting a specific traffic sign — say a red octagonal STOP sign — where all instances look nearly identical. In this case, the Dalal-Triggs single-template detector would produce a sharp, precise template (no averaging-induced blurriness) because all training examples look the same. Meanwhile, the Exemplar-SVM approach would train hundreds of nearly identical SVMs, each with only one positive, leading to higher computational cost with no benefit. Furthermore, with few positive exemplars, each Exemplar-SVM has extremely limited information (one positive point) and relies entirely on the quality of the negative mining and calibration. A standard Dalal-Triggs detector, which pools all positives together, would have more statistical power to estimate the decision boundary. The paper's Table 1 shows that for some categories (like 'bottle' with mAP 0.102 vs Dalal-Triggs 0.104), the Exemplar-SVM does not meaningfully outperform the baseline, likely because bottles have less appearance variation than categories like 'person' or 'cat'.